### Import Libraries

In [ ]:
import re
import torch
import joblib
import contractions
import numpy as np
from textblob import TextBlob
from lib_file import lib_path
from transformers import GPT2Model, GPT2Tokenizer

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

### Model Loading

In [ ]:
# Load model
lr_model = joblib.load("models/lr_model.pkl")

# Re-save to fix sklearn version mismatch warning
joblib.dump(lr_model, "models/lr_model.pkl")

# Load GPT-2
gpt2_model = GPT2Model.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def clean_text(text):
    text = contractions.fix(text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    blob = TextBlob(text)
    words = [word.lemmatize() for word in blob.words if word.lower() not in stop_words]

    return " ".join(words).lower()

In [ ]:
# Feature Extraction (GPT-2)
def extract_features(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = gpt2_model(**inputs)
    
    # Mean pooling
    embedding = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
    return embedding.reshape(1, -1)

***Prediction***

In [ ]:
def predict_sentiment(review):
    cleaned = clean_text(review)
    features = extract_features(cleaned)
    
    prediction = lr_model.predict(features)[0]
    
    label_map = {0: "Positive", 1: "Negative"}
    return label_map[prediction]

In [ ]:
review = input("Enter a movie review: ")
result = predict_sentiment(review)
print(f"Predicted Sentiment: {result}")

---